In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100  # MB
from IPython.display import display
%matplotlib inline
%matplotlib notebook
pip install pillow
from IPython.display import display, HTML
%matplotlib agg
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML
# =====================================
# 1. Himmelblau Function
# =====================================
def loss(w1, w2):
    return (w1**2 + w2 - 11)**2 + (w1 + w2**2 - 7)**2
def gradients(w1, w2):
    d1 = 4*w1*(w1**2 + w2 - 11) + 2*(w1 + w2**2 - 7)
    d2 = 2*(w1**2 + w2 - 11) + 4*w2*(w1 + w2**2 - 7)
    return d1, d2
#=====================================
# 2. Hyperparameters
# =====================================
lr = 0.01
iters = 200
momentum = 0.9
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
w1_init = 0
w2_init = 0
# =====================================
# 3. Optimizers
# =====================================
def GD():
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        w1 -= lr * g1
        w2 -= lr * g2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
              return path, losses

def SGD():
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        noise = np.random.normal(0, 0.15)
        w1 -= lr * (g1 + noise)
        w2 -= lr * (g2 + noise)
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def MiniBatch():
    w1, w2 = w1_init, w2_init
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        g1 += np.random.normal(0, 0.07)
        g2 += np.random.normal(0, 0.07)
        w1 -= lr * g1
        w2 -= lr * g2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def Nesterov():
    w1, w2 = w1_init, w2_init
    v1, v2 = 0, 0
    path, losses = [], []
    for _ in range(iters):
        lw1 = w1 - momentum*v1
        lw2 = w2 - momentum*v2
        g1, g2 = gradients(lw1, lw2)
        v1 = momentum*v1 + lr*g1
        v2 = momentum*v2 + lr*g2
        w1 -= v1
        w2 -= v2
        path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def Adagrad():
    w1, w2 = w1_init, w2_init
    h1, h2 = 0, 0
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        h1 += g1**2
        h2 += g2**2
        w1 -= lr*g1/(np.sqrt(h1)+eps)
        w2 -= lr*g2/(np.sqrt(h2)+eps)
                  path.append((w1, w2))
        losses.append(loss(w1, w2))
    return path, losses
def RMSProp():
    w1, w2 = w1_init, w2_init
    h1, h2 = 0, 0
    rho = 0.9
    path, losses = [], []
    for _ in range(iters):
        g1, g2 = gradients(w1, w2)
        h1 = rho*h1 + (1-rho)*g1**2
        h2 = rho*h2 + (1-rho)*g2**2
        w1 -= lr*g1/(np.sqrt(h1)+eps)
        w2 -= lr*g2/(np.sqrt(h2)+eps)

        path.append((w1, w2))
        losses.append(loss(w1, w2))

    return path, losses

def Adam():

    w1, w2 = w1_init, w2_init

    m1 = m2 = v1 = v2 = 0

    path, losses = [], []

    for t in range(1, iters+1):

        g1, g2 = gradients(w1, w2)

        m1 = beta1*m1 + (1-beta1)*g1
        m2 = beta1*m2 + (1-beta1)*g2

        v1 = beta2*v1 + (1-beta2)*g1**2
        v2 = beta2*v2 + (1-beta2)*g2**2

        m1h = m1/(1-beta1**t)
        m2h = m2/(1-beta1**t)

        v1h = v1/(1-beta2**t)
        v2h = v2/(1-beta2**t)

        w1 -= lr*m1h/(np.sqrt(v1h)+eps)
        w2 -= lr*m2h/(np.sqrt(v2h)+eps)

        path.append((w1, w2))
        losses.append(loss(w1, w2))

    return path, losses

# =====================================
# 4. Loss Surface
# =====================================

x = np.linspace(-6, 6, 120)
y = np.linspace(-6, 6, 120)

X, Y = np.meshgrid(x, y)
Z = loss(X, Y)

# =====================================
# 5. Animation Functions
# =====================================

def animate_2D_inline(name, path):

    fig, ax = plt.subplots()

    ax.contour(X, Y, Z, 50)
    ax.set_title(f"{name} - 2D Contour")

    line, = ax.plot([], [], 'r-', lw=2)
    point, = ax.plot([], [], 'ro')

    def update(i):

        xs = [p[0] for p in path[:i]]
        ys = [p[1] for p in path[:i]]

        if len(xs) == 0:
            return line, point

        line.set_data(xs, ys)
        point.set_data([xs[-1]], [ys[-1]])

        return line, point

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=range(1, len(path)+1),
        interval=50,
        blit=True
    )

    plt.close(fig)

    return HTML(anim.to_jshtml())

def animate_3D_inline(name, path):

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.7)

    ax.set_title(f"{name} - 3D Surface")

    point, = ax.plot([], [], [], 'ro', markersize=6)

    def update(i):

        w1, w2 = path[i-1]
        z = loss(w1, w2)

        point.set_data([w1], [w2])
        point.set_3d_properties([z])

        return point,

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=range(1, len(path)+1),
        interval=80,
        blit=True
    )

    plt.close(fig)

    return HTML(anim.to_jshtml())

def plot_loss_inline(name, losses):

    fig, ax = plt.subplots()

    ax.plot(losses, lw=2)

    ax.set_title(f"{name} - Loss Curve")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(True)

    plt.close(fig)

    display(fig)

# =====================================
# 6. Run Optimizers
# =====================================
path, l = GD()
display(animate_2D_inline("GD", path))
display(animate_3D_inline("GD", path))
plot_loss_inline("GD", l)

path, l = SGD()
display(animate_2D_inline("SGD", path))
display(animate_3D_inline("SGD", path))
plot_loss_inline("SGD", l)

path, l = MiniBatch()
display(animate_2D_inline("MiniBatch", path))
display(animate_3D_inline("MiniBatch", path))
plot_loss_inline("MiniBatch", l)

path, l = Nesterov()
display(animate_2D_inline("Nesterov", path))
display(animate_3D_inline("Nesterov", path))
plot_loss_inline("Nesterov", l)

path, l = Adagrad()
display(animate_2D_inline("Adagrad", path))
display(animate_3D_inline("Adagrad", path))
plot_loss_inline("Adagrad", l)

path, l = RMSProp()
display(animate_2D_inline("RMSProp", path))
display(animate_3D_inline("RMSProp", path))
plot_loss_inline("RMSProp", l)

path, l = Adam()
display(animate_2D_inline("Adam", path))
display(animate_3D_inline("Adam", path))
plot_loss_inline("Adam", l) 
